# gemma4:e4b — the actual final document, sample by sample

This shows exactly what you'd see if you opened the finished output file for
each of the 10 sample documents — title, then each section with its question
and answer. Anything that doesn't match the human-written answer key it's
checked against is highlighted, with a plain-English note explaining why.

**"Final" and "Path B" here mean the same thing**: the pipeline's last stage,
after a small automatic fix-up step runs (filling in a question that was left
blank, using the section heading above it). That's the file this notebook
reads — nothing here is the model's raw, unfixed output.


**How to read each sample:**

Each sample is shown the way the finished document actually reads. Most items
have nothing extra next to them — that means they match the answer key exactly.

An item shown with a light amber highlight didn't match cleanly, and has a short
note underneath explaining why — for example, a single word differs, the text
was filed under the wrong heading type, or it's the same content already counted
elsewhere in the document.

A pink box at the end of a sample, if one appears, lists anything the answer key
expects that isn't in the document at all.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import extract_gold, _match_structured, tokenize, containment
from dmpbridge.evaluation.annotation_rules import resolve_new_gt_path, convert_tag_to_final

MODEL, TAG = 'gemma4:e4b', 'gemma4-e4b_pdfplumber_whole_doc'
SAMPLES = range(1, 11)
NEAR_MISS_FLOOR = 0.85   # containment above this -> "near-miss", show the exact
                         # difference; below it -> genuinely not in the answer key

if not P.final_path(TAG, 1).exists():
    convert_tag_to_final(TAG)


def closest_gold(text, label, gold):
    """Best-matching gold item of the same label, by word containment, or
    (None, 0.0) if this label has no gold items at all."""
    pt = tokenize(text)
    best_text, best_score = None, 0.0
    for gtext, glabel in gold:
        if glabel != label:
            continue
        score = containment(pt, tokenize(gtext))
        if score > best_score:
            best_text, best_score = gtext, score
    return best_text, best_score


def sample_status(n):
    """(status_lookup, missing) for one sample's final JSON vs its answer key.

    status_lookup maps (text, label) -> a small dict describing what, if
    anything, is wrong with that item ({'kind': 'match'} if it's fine).
    missing lists answer-key items nothing in the document matched. Path B
    only: dedup_question_title=False is Path B's own setting, since it exists
    specifically to measure the fill-in step Path A skips.

    A predicted item that doesn't match anything gets one of three different
    kinds, not one generic flag: a near-miss of some gold item (very high word
    overlap, just not 100%) names the exact word(s) that differ; a near-miss
    with zero differing words is content already claimed by a better-fitting
    part of the prediction elsewhere (duplicate); anything else has no close
    gold counterpart at all (not_in_key).
    """
    gold = extract_gold(resolve_new_gt_path(n), dedup_question_title=False)
    records, no_gold = _match_structured(P.final_path(TAG, n), gold, dedup_question_title=False)

    status = {}
    for r in records:
        if r['pred_text'] is not None:
            key = (r['pred_text'].strip(), r['pred_label'])
            if r['pred_label'] == r['gold_label']:
                status[key] = {'kind': 'match'}
            else:
                status[key] = {'kind': 'wrong_label', 'gold_label': r['gold_label']}
    for text, label in no_gold:
        gold_text, score = closest_gold(text, label, gold)
        if gold_text is None or score < NEAR_MISS_FLOOR:
            status[(text.strip(), label)] = {'kind': 'not_in_key'}
            continue
        missing_words = sorted(tokenize(text) - tokenize(gold_text))
        if missing_words:
            status[(text.strip(), label)] = {
                'kind': 'near_miss', 'score': score, 'words': missing_words}
        else:
            # Every word IS in the answer key, but this exact answer-key item was
            # already claimed by a different, better-fitting piece of the
            # prediction — so this is leftover/duplicate content, not a word
            # mismatch (e.g. one long answer got split into overlapping pieces).
            status[(text.strip(), label)] = {'kind': 'duplicate'}

    missing = [(r['gold_label'], r['gold_text']) for r in records if r['pred_text'] is None]
    return status, missing


## Sample by sample


In [2]:
import html
from IPython.display import display, HTML

LABEL_NAMES = {
    'title': 'Document title',
    'section.title': 'Section heading',
    'section.description': 'Section description',
    'question.text': 'Question',
    'answer.text': 'Answer',
}


def word_list(words):
    """['sub'] -> '“sub”'; ['a', 'b'] -> '“a” and “b”'."""
    quoted = [f'\u201c{w}\u201d' for w in words]
    if len(quoted) == 1:
        return quoted[0]
    return ', '.join(quoted[:-1]) + ' and ' + quoted[-1]


def describe(flag):
    """A plain-English sentence for one flag dict, or None for a clean match."""
    kind = flag['kind']
    if kind == 'match':
        return None
    if kind == 'wrong_label':
        return (f"This text is right, but the answer key labels it as "
                f"{LABEL_NAMES[flag['gold_label']].lower()}, not this.")
    if kind == 'near_miss':
        n = len(flag['words'])
        return (f"Almost an exact match to the answer key ({flag['score']:.0%}) \u2014 only "
                f"the word{'s' if n > 1 else ''} {word_list(flag['words'])} "
                f"{'are' if n > 1 else 'is'} different. Often a small PDF extraction quirk, "
                f"not a real mistake.")
    if kind == 'not_in_key':
        return ("This text doesn't appear anywhere in the answer key \u2014 it looks like "
                "an extra item that shouldn't be here.")
    if kind == 'duplicate':
        return ("These exact words already appear earlier in this document under a "
                "different heading \u2014 this looks like the same content counted twice.")


def note_for(status, text, label):
    flag = status.get((text.strip(), label))
    note = describe(flag) if flag else None
    if label == 'question.text' and text[:1].isalpha() and text[:1].islower():
        lower_note = ("This question starts with a lowercase letter, which often means "
                      "it's a leftover piece of a sentence rather than a real, separate "
                      "question.")
        note = f'{note} {lower_note}' if note else lower_note
    return note


def render_item(label_name, text, note, indent):
    text_esc = html.escape(text) if text else '(empty)'
    label_html = (f'<div style="font-size:0.72em; text-transform:uppercase; '
                  f'letter-spacing:0.04em; color:#888; margin:10px 0 2px {indent}px;">'
                  f'{label_name}</div>')
    box_style = f'margin:0 0 4px {indent}px; padding:6px 10px; border-radius:4px;'
    if note is None:
        return label_html + f'<div style="{box_style}">{text_esc}</div>'
    box_style += ' background:#fff8e1; border-left:3px solid #f0ad4e;'
    note_html = (f'<div style="margin-top:4px; font-size:0.85em; color:#7a5b00;">'
                 f'Note: {html.escape(note)}</div>')
    return label_html + f'<div style="{box_style}">{text_esc}{note_html}</div>'


def render_sample(n):
    status, missing = sample_status(n)
    s4 = json.loads(P.final_path(TAG, n).read_text(encoding='utf-8'))
    t = s4['narrative']['template']

    parts, counts = [], {'total': 0, 'flagged': 0}

    def item(label_key, text, indent):
        counts['total'] += 1
        note = note_for(status, text, label_key)
        if note is not None:
            counts['flagged'] += 1
        parts.append(render_item(LABEL_NAMES[label_key], text, note, indent))

    item('title', t.get('title', ''), 0)
    for s in t.get('section', []):
        item('section.title', s.get('title', ''), 0)
        for q in s.get('question', []):
            q_text = q.get('text', '')
            ans = q.get('answer', {}).get('json', {}).get('answer', '')
            item('question.text', q_text, 20)
            item('answer.text', ans, 20)

    total, flagged = counts['total'], counts['flagged']
    summary = f'{total - flagged} of {total} items match the answer key exactly.'
    if flagged:
        summary += f' {flagged} flagged below, with a plain-English note.'

    body = (
        f'<div style="font-size:1.3em; font-weight:700; margin:26px 0 2px;">Sample {n}</div>'
        f'<div style="color:#555; margin-bottom:8px;">{summary}</div>'
        + ''.join(parts)
    )

    if missing:
        rows = ''.join(
            f'<div style="margin:4px 0;"><b>{LABEL_NAMES[lab]}:</b> {html.escape(txt)}</div>'
            for lab, txt in missing
        )
        body += (
            '<div style="margin-top:14px; padding:10px; border-radius:4px; '
            'background:#fdecea; border-left:3px solid #d9534f;">'
            '<b>Not found in this document</b> \u2014 the answer key expects these but '
            f'none of them appear here:{rows}</div>'
        )

    return (
        '<div style="font-family: -apple-system, Segoe UI, Helvetica, Arial, sans-serif; '
        f'font-size:15px; line-height:1.5; color:#222; max-width:820px;">{body}</div>'
    )


for n in SAMPLES:
    display(HTML(render_sample(n)))


## Takeaways

- Most documents match the answer key cleanly, item for item — no highlight at all.
- Where something is highlighted, the note says exactly why: **wrong label** means
  the text was found but filed under the wrong heading type; **doesn't appear in
  the answer key** usually means the document was split differently than the
  answer key expects (a heading glued onto its answer, or one long answer cut
  into pieces); a lowercase-start note is a quick way to spot a question that's
  actually a stray sentence fragment.
- A near-miss can look identical to the eye but technically not match — usually a
  tiny extraction artifact (a line-wrapped word like "sub-study" coming through
  with an extra space where the PDF wrapped the line) rather than anything the
  model got wrong.
